# Tokenizer 分词：从文本到模型输入

Tokenizer（分词器）把文本转换为模型可处理的 token ID 序列，也可将 ID 解码回文本。它不只是按空格切词：规范化、预切分、子词算法、特殊 token、截断、填充和 mask 都会影响模型实际接收到的输入。

本文由浅入深介绍字符级、词级、子词级、BPE、WordPiece、Unigram、预处理 pipeline、特殊 token、编码解码、批处理、Chat Template、上下文窗口与推理成本。示例以 Python、PyTorch 2.x 和 Hugging Face 常见 API 为例；具体行为应以目标模型的 tokenizer 配置为准。

> 重要原则：模型权重必须与训练时使用的 tokenizer、词表、特殊 token 定义和 ID 映射匹配。词表大小相同不代表 ID 含义相同。

> 大语言模型并不是直接理解文字，而是先通过 Tokenizer 将文本转换成 Token ID，再将 Token ID 转换成 Embedding，输入 Transformer。

> 原始文本 -> 规范化 -> 预切分 -> 子词切分/查词表 -> token ID -> 添加结构 token -> 截断/填充/mask -> embedding -> 模型


In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device: {device}")

In [ ]:

## 1. 从文本到模型输入 最简单示例


# 原始输入文本
text = "I like apples."
# token 分词
tokens = ["I", "like", "apples", "." ]
# 构造词汇表
vocab = {token: i for i, token in enumerate(tokens)}
# 通过词汇表获取对应的 token_ids
token_ids = [vocab[token] for token in tokens ]
# 通过 token_ids 获取对应的 embedding 向量
embedding = nn.Embedding(len(token_ids), 16)
embeds = embedding(torch.LongTensor(token_ids))
# 将 embedding 出来的向量直接输入语言模型
print(embeds.shape)


In [ ]:
## 2. Token、词表与 ID

# - Token：字符、单词、子词、字节片段或结构标记。
# - Vocabulary（词表）：token 到整数 ID 的映射。
# - Token ID：embedding 表索引，没有大小或语义关系。
# - 序列长度 L：token 个数，不等于字符数或空格分隔的单词数。

# 不同 tokenizer 的 ID 不能混用。ID 20 不比 ID 10 更“大”，它只是另一个 embedding 行。

vocab = {"<UNK>": 0, "#": 1, "@": 2}
token_ids = [vocab["#"], vocab["@"]]
print(token_ids)


In [ ]:
## 3. 字符级分词
# 字符级 tokenizer 把文本拆成字符，也可能按 Unicode code point、字节或 grapheme cluster 切分。它词表小、较容易覆盖新词，但序列长且单个字符语义弱。
text = "字符级分词（Character-level Tokenization）：将文本拆分为单个字符，每个字符作为一个独立的 Token。"

# 统计每一个字符出现的次数，并且从高到低排序
from collections import Counter
counter = Counter(text) # 统计每一个字符出现的次数
char_set = sorted(counter.items(), key=lambda x: x[1], reverse=True) # 按照词频从高到低排序
for id, (token, count) in enumerate(char_set):
    print(f"id: {id} -> {token}: {count}")
# print(char_set)
char_to_id = {token: id for id, (token, count) in enumerate(char_set)}
input_text = "字符级分词"
input_token_ids = [char_to_id[input] for id, input in enumerate(input_text)]
input_token_ids

In [ ]:
## 4. 词级分词
# 词级方法按完整词或规则切分。英语常利用空格和标点；中文没有天然空格边界，需要语言规则或中文分词器。有限词表会使未登录词变成 UNK。
import re
text = "Tokenizer is useful, isn't it?"
tokens = re.findall(r"\w+|[^\w\s]", text.lower())
print(tokens)
# 之后再去构造词汇表

In [ ]:
## 5. 子词分词
# 子词（subword）让常见片段成为一个 token，罕见词拆成多个片段。例如 unhappiness 可能拆为 un、happi、ness，实际结果取决于词表。它折中词表规模、序列长度与 OOV 覆盖。
# 子词 tokenizer 通常需要能覆盖任意输入的基础单位，如字符或 byte-level 字母表。具体覆盖范围由算法和配置决定。


In [ ]:
## 6. BPE
# BPE 的核心思想非常简单：不断统计语料中相邻 Token 对的出现次数，将出现频率最高的相邻 Token 合并成一个新的 Token，直到达到指定的词表大小或合并次数。
# 我们分成三个核心步骤：
# 1. get_pair_freq()：统计相邻 Token 对的出现频率。
# 2. merge_pair()：将指定的相邻 Token 对合并。
# 3. train_bpe()：循环执行统计和合并，生成最终词表和合并规则。
